# Графовый анализ научных публикаций

Собрать корпус статей о GraphRAG, построить графы терминов и публикаций, выделить сообщества и сравнить варианты поиска.

[Описание и данные](README.md) · [История эксперимента](results/recorded_metrics.json)

**Запуск:** Python 3.11 или 3.12. Полное обучение требует данных и подходящего GPU. Выводы очищены; численные результаты ниже появятся после выполнения.

## Подготовка окружения

Ячейка устанавливает зависимости проекта. Локально рекомендуется отдельное виртуальное окружение.

In [ ]:
%pip -q install -r https://raw.githubusercontent.com/Xokage9/ml-portfolio/main/projects/06_graph_search/requirements.txt


In [ ]:
import os
os.environ['OMP_NUM_THREADS']='2'
os.environ['OPENBLAS_NUM_THREADS']='2'
os.environ['MKL_NUM_THREADS']='2'
from threadpoolctl import threadpool_limits
threadpool_limits(limits=2)


In [ ]:
!python -m spacy download en_core_web_sm -q

import re
import time
import math
import random
import itertools
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import spacy

pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
random.seed(42); np.random.seed(42)


## 1. Сбор данных через arXiv API

Ищем статьи, в аннотации которых есть «retrieval-augmented generation» и graph или GraphRAG. Берём публикации с 2020 года по 06.11.2026.

Загружаем по 200 записей с паузой 3.5 с, повторяем неудачные запросы и убираем дубли. Корпус сохраняем в CSV для следующих запусков.


In [ ]:
API = "https://export.arxiv.org/api/query"
QUERY = 'abs:"retrieval-augmented generation" AND (abs:graph OR abs:graphrag)'
NS = {"a": "http://www.w3.org/2005/Atom", "arxiv": "http://arxiv.org/schemas/atom",
      "os": "http://a9.com/-/spec/opensearch/1.1/"}
PAGE, PAUSE, RETRIES = 200, 3.5, 5
CSV = "graph_rag_arxiv.csv"
CUTOFF = "2026-11-06"


def fetch_page(start):
    params = urllib.parse.urlencode({"search_query": QUERY, "start": start, "max_results": PAGE,
                                     "sortBy": "submittedDate", "sortOrder": "descending"})
    for attempt in range(RETRIES):
        try:
            req = urllib.request.Request(f"{API}?{params}", headers={"User-Agent": "itmo-lab1/1.0"})
            with urllib.request.urlopen(req, timeout=60) as r:
                root = ET.fromstring(r.read())
            entries = root.findall("a:entry", NS)
            total = int(root.find("os:totalResults", NS).text)
            if entries or start >= total:
                return entries, total
            raise RuntimeError("пустая страница")
        except Exception as e:
            wait = PAUSE * 2 ** attempt
            print(f"  start={start}: {e}; повтор через {wait:.0f} c")
            time.sleep(wait)
    raise RuntimeError(f"не удалось получить страницу start={start}")


def parse(e):
    text = lambda tag: " ".join((e.find(tag, NS).text or "").split())
    return {"arxiv_id": text("a:id").rsplit("/", 1)[-1].split("v")[0],
            "title": text("a:title"),
            "authors": "; ".join(a.find("a:name", NS).text for a in e.findall("a:author", NS)),
            "abstract": text("a:summary"),
            "published": text("a:published")[:10],
            "primary_category": e.find("arxiv:primary_category", NS).get("term"),
            "categories": " ".join(c.get("term") for c in e.findall("a:category", NS)),
            "url": text("a:id")}


def collect():
    rows, start, total = [], 0, None
    while total is None or start < total:
        entries, total = fetch_page(start)
        rows += [parse(e) for e in entries]
        print(f"получено {len(rows)} из {total}")
        start += PAGE
        time.sleep(PAUSE)
    d = pd.DataFrame(rows).drop_duplicates("arxiv_id").reset_index(drop=True)
    d["published"] = pd.to_datetime(d["published"])
    return d[(d.published.dt.year >= 2020) & (d.published < CUTOFF)].reset_index(drop=True)


try:
    df = pd.read_csv(CSV, parse_dates=["published"], dtype={"arxiv_id":str})
    print("загружено из файла")
except FileNotFoundError:
    df = collect()
    df.to_csv(CSV, index=False)
assert len(df)>=300
print(f"статей: {len(df)}, период: {df.published.min():%d.%m.%Y} – {df.published.max():%d.%m.%Y}")
df.head(3)


## 2. Разведочный анализ корпуса

### 2.1. Описание данных


In [ ]:
df["n_authors"] = df.authors.str.split("; ").str.len()
df["abstract_words"] = df.abstract.str.split().str.len()
df["title_words"] = df.title.str.split().str.len()
quality = pd.Series({
    "статей": len(df),
    "пустых названий / аннотаций": int((df.title.str.len() == 0).sum() + (df.abstract.str.len() == 0).sum()),
    "дублей по arxiv_id": int(df.arxiv_id.duplicated().sum()),
    "дублей по названию": int(df.title.str.lower().duplicated().sum()),
    "уникальных авторов": df.authors.str.split("; ").explode().nunique(),
    "основных категорий": df.primary_category.nunique(),
})
display(quality.to_frame("значение"))
n0 = len(df)
df = df[~df.title.str.lower().duplicated(keep="last")].reset_index(drop=True)
print(f"удалено дублей по названию: {n0 - len(df)}, осталось статей: {len(df)}")
df[["n_authors", "abstract_words", "title_words"]].describe().T.round(1)


In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 8))
month = df.set_index("published").resample("QE").size()
ax[0, 0].bar(range(len(month)), month.values, color="#4a6fa5")
ax[0, 0].set_xticks(range(0, len(month), 2), [f"{d.year}-Q{(d.month - 1) // 3 + 1}" for d in month.index][::2],
                    rotation=45, ha="right")
ax[0, 0].set_title("Публикаций по кварталам")

top_cat = df.primary_category.value_counts().head(10)
top_cat[::-1].plot.barh(ax=ax[0, 1], color="#4a6fa5"); ax[0, 1].set_title("Основная категория arXiv (топ-10)")

ax[1, 0].hist(df.abstract_words, bins=40, color="#4a6fa5")
ax[1, 0].set_title("Длина аннотации, слов"); ax[1, 0].set_xlabel("слов")

ax[1, 1].hist(df.n_authors.clip(upper=20), bins=20, color="#4a6fa5")
ax[1, 1].set_title("Авторов на статью (20 = 20 и больше)")
plt.tight_layout(); plt.show()

authors = df.authors.str.split("; ").explode()
print("самые продуктивные авторы:")
print(authors.value_counts().head(8).to_string())
by_year = df.published.dt.year.value_counts().sort_index()
print("\nстатей по годам:", by_year.to_dict())


Проверяем пропуски, дубли и число статей. Дальше работаем с очищенным корпусом.


### 2.2. Закон Ципфа и закон Хипса

Ципф: частота слова убывает с его рангом, $f(r) \approx C/r^s$.

Хипс: словарь растёт с объёмом текста, $V(N) \approx KN^\beta$. Проверяем обе зависимости на нашем корпусе.


In [ ]:
TOKEN_RE = re.compile(r"[a-z][a-z\-]*[a-z]|[a-z]")
docs_raw = (df.title + ". " + df.abstract).str.lower()
tokens_per_doc = [TOKEN_RE.findall(t) for t in docs_raw]
all_tokens = list(itertools.chain.from_iterable(tokens_per_doc))
freq = Counter(all_tokens)
f = np.array(sorted(freq.values(), reverse=True), float)
r = np.arange(1, len(f) + 1)
mask = (r >= 10) & (r <= 2000)
s, logC = np.polyfit(np.log(r[mask]), np.log(f[mask]), 1)

seen, V, N, n = set(), [], [], 0
for toks in tokens_per_doc:
    for t in toks:
        n += 1; seen.add(t)
    N.append(n); V.append(len(seen))
N, V = np.array(N), np.array(V)
beta, logK = np.polyfit(np.log(N), np.log(V), 1)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].loglog(r, f, ".", ms=3, color="#4a6fa5", label="корпус")
ax[0].loglog(r, np.exp(logC) * r ** s, color="#c0392b", label=f"аппроксимация: наклон {s:.2f}")
ax[0].loglog(r, f[0] / r, "--", color="#999999", label="идеальный Ципф: наклон −1")
for w, (rr, ff) in zip([w for w, _ in freq.most_common(6)], zip(r[:6], f[:6])):
    ax[0].annotate(w, (rr, ff), fontsize=8, xytext=(4, 2), textcoords="offset points")
ax[0].set_xlabel("ранг"); ax[0].set_ylabel("частота"); ax[0].set_title("Закон Ципфа"); ax[0].legend()
ax[1].plot(N, V, color="#4a6fa5", label="корпус")
ax[1].plot(N, np.exp(logK) * N ** beta, "--", color="#c0392b", label=f"$V = {np.exp(logK):.1f} N^{{{beta:.2f}}}$")
ax[1].set_xlabel("токенов прочитано"); ax[1].set_ylabel("размер словаря"); ax[1].set_title("Закон Хипса")
ax[1].legend()
plt.tight_layout(); plt.show()

print(f"токенов: {len(all_tokens):,}, уникальных: {len(freq):,}")
print(f"Ципф: наклон {s:.2f}; Хипс: K = {np.exp(logK):.1f}, β = {beta:.2f}")
top = freq.most_common(100)
share_top = sum(c for _, c in top) / len(all_tokens)
hapax = sum(1 for c in freq.values() if c == 1) / len(freq)
print(f"100 самых частых слов дают {share_top:.0%} всех токенов; слов, встретившихся один раз: {hapax:.0%} словаря")
print("топ-25:", ", ".join(w for w, _ in freq.most_common(25)))


По графикам смотрим, насколько данные соответствуют законам. Коэффициенты зависят от корпуса и обработки текста.


## 3. Выделение ключевых слов

Приводим варианты терминов к одному виду: например, Graph RAG → graphrag. Из названий и аннотаций извлекаем именные группы длиной 1–3 слова, приводим слова к начальной форме и убираем лишние.

Для статьи оставляем до 10 фраз с наибольшим TF-IDF; фразы из названия учитываем с двойным весом. Берём фразы, встречающиеся минимум в 5 статьях, и исключаем слишком общие — те, что есть более чем в 40% корпуса.


In [ ]:
nlp = spacy.load("en_core_web_sm", disable=["ner"])

NORMALIZE = [
    (r"graph[\s-]*rag\b|graphrag", " GRAPHRAGTERM "),
    (r"retrieval[\s-]*augmented[\s-]*generation(s)?|\brag\b", " RAGTERM "),
    (r"large[\s-]*language[\s-]*models?|\bllms?\b", " LLMTERM "),
    (r"knowledge[\s-]*graphs?|\bkgs?\b", " KGTERM "),
    (r"question[\s-]*answering|\bqa\b", " QATERM "),
    (r"multi[\s-]*hop", "multihop"),
    (r"multi[\s-]*modal", "multimodal"),
    (r"fine[\s-]*tun(e|ing|ed)", "finetuning"),
    (r"chain[\s-]*of[\s-]*thoughts?|\bcot\b", " COTTERM "),
    (r"\bvector[\s-]*databases?\b", "vector database"),
]
RESTORE = {"ragterm": "rag", "graphragterm": "graphrag", "llmterm": "llm", "kgterm": "knowledge graph",
           "qaterm": "question answering", "cotterm": "chain of thought"}

GENERIC = set((
    "approach method methods paper work study result performance framework experiment "
    "task model system technique way issue problem challenge limitation effectiveness efficiency "
    "state art baseline improvement benchmark use number variety range set type form level "
    "existing new novel various different large small high low key significant extensive "
    "comprehensive effective efficient robust strong impressive substantial recent current "
    "contribution advantage capability ability process component aspect manner context information "
    "datum data result analysis evaluation application domain field area case scenario setting "
    "need lack gap source access quality accuracy amount part step stage end base "
    "request absolute availability premise name extended response output input object product pair "
    "final strength class critical superior average progress open relevant gain question answer time text "
    "language design change combination factor construction representative boundary condition conclusion "
    "feasibility dataset experiment outcome property detail point fact term size cost generation potential "
    "finding user integration insight percentage sub value impact mechanism development advancement "
    "architecture shot test creation phase business hundred place list unit chain perspective evolution "
    "candidate collection space distribution paradigm core score loop validation synthesis stake content "
    "complexity structure real world version option example instance item element feature tool advance "
    "demand practice principle idea concept attempt effort goal purpose objective ii iii iv addition "
    "absence opportunity groundwork support selection figure label dimension promise role depth foundation "
    "innovation environment service solution difficulty experience learning strategy resource").split())


def normalize_text(t):
    t = t.lower()
    for pat, rep in NORMALIZE:
        t = re.sub(pat, rep, t)
    return t


def clean_phrase(span):
    toks = [w for w in span if w.is_alpha or w.lower_.endswith("term")]
    lem = [RESTORE.get(w.lower_, w.lemma_.lower()) for w in toks]
    keep = [(w, l) for w, l in zip(toks, lem)
            if not (w.is_stop or l in GENERIC or len(l) < 2) or l in RESTORE.values()]
    while keep and (keep[0][0].pos_ not in ("NOUN", "PROPN", "ADJ") and keep[0][1] not in RESTORE.values()):
        keep = keep[1:]
    words = list(dict.fromkeys(" ".join(l for _, l in keep).split()))
    if not 1 <= len(words) <= 3:
        return None
    if len(words) == 1 and words[0] not in RESTORE.values() and keep[0][0].pos_ not in ("NOUN", "PROPN"):
        return None
    phrase = " ".join(words)
    return PHRASE_SYN.get(phrase, phrase)


PHRASE_SYN = {"standard rag": "vanilla rag", "traditional rag": "vanilla rag", "conventional rag": "vanilla rag",
              "naive rag": "vanilla rag", "baseline rag": "vanilla rag", "real world": None,
              "graph rag": "graphrag", "graph retrieval augment generation": "graphrag",
              "graph base retrieval": "graph retrieval", "rag system": "rag", "rag framework": "rag",
              "rag method": "rag", "kg": "knowledge graph"}


def candidates(doc):
    out = []
    for ch in doc.noun_chunks:
        p = clean_phrase(ch)
        if p:
            out.append(p)
    return out


t0 = time.time()
titles = list(nlp.pipe((normalize_text(t) for t in df.title), batch_size=64))
abstracts = list(nlp.pipe((normalize_text(t) for t in df.abstract), batch_size=64))
cand_title = [candidates(d) for d in titles]
cand_abs = [candidates(d) for d in abstracts]
print(f"разбор spaCy: {time.time() - t0:.0f} c")
print("пример кандидатов для первой статьи:", (cand_title[0] + cand_abs[0])[:15])


In [ ]:
N_DOCS = len(df)
doc_freq = Counter(itertools.chain.from_iterable(set(a + b) for a, b in zip(cand_title, cand_abs)))
too_common = {p for p, c in doc_freq.items() if c / N_DOCS > 0.40}
print("исключены как слишком частые (>40% статей):",
      {p: f"{doc_freq[p] / N_DOCS:.0%}" for p in sorted(too_common, key=lambda p: -doc_freq[p])})

TOP_K, MIN_KW_DF = 10, 5


def keywords_for(i):
    tf = Counter(cand_abs[i])
    for p in cand_title[i]:
        tf[p] += 2
    scores = {p: (1 + math.log(c)) * math.log(N_DOCS / doc_freq[p])
              for p, c in tf.items() if doc_freq[p] >= MIN_KW_DF and p not in too_common}
    return [p for p, _ in sorted(scores.items(), key=lambda x: -x[1])[:TOP_K]]


df["keywords"] = [keywords_for(i) for i in range(N_DOCS)]
df["n_keywords"] = df.keywords.str.len()
kw_freq = Counter(itertools.chain.from_iterable(df.keywords))
ngram = Counter(len(p.split()) for p in kw_freq)
print(f"\nуникальных ключевых фраз: {len(kw_freq)}; по длине: "
      f"униграммы {ngram[1]}, биграммы {ngram[2]}, триграммы {ngram[3]}")
print(f"ключевых фраз на статью: в среднем {df.n_keywords.mean():.1f}")
df[["title", "keywords"]].head(8)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
top_kw = pd.Series(kw_freq).sort_values(ascending=False).head(30)
top_kw[::-1].plot.barh(ax=ax[0], color="#4a6fa5")
ax[0].set_title("30 самых частых ключевых фраз (число статей)")
by_len = {n: pd.Series({p: c for p, c in kw_freq.items() if len(p.split()) == n}).nlargest(8) for n in (2, 3)}
txt = "\n".join([f"биграммы:"] + [f"  {p} — {c}" for p, c in by_len[2].items()]
                + ["", "триграммы:"] + [f"  {p} — {c}" for p, c in by_len[3].items()])
ax[1].axis("off"); ax[1].text(0, 1, txt, va="top", family="monospace", fontsize=10)
ax[1].set_title("Самые частые многословные ключевые фразы")
plt.tight_layout(); plt.show()
df.to_csv("graph_rag_arxiv_keywords.csv", index=False)


## 4. Граф ключевых слов

Вершины — фразы, встречающиеся минимум в 5 статьях. Соединяем две фразы, если они вместе встречаются хотя бы в двух статьях.

Вес связи нормируем по частотам: $w_{ab}=c_{ab}/\sqrt{c_a c_b}$. Для каждой вершины выбираем 10 сильнейших связей. Сравниваем способы взвешивания по модулярности.


In [ ]:
MIN_DF, MIN_C, TOP_NB = 5, 2, 10
vocab = {p for p, c in kw_freq.items() if c >= MIN_DF}
pair_c = Counter()
for kws in df.keywords:
    pair_c.update(itertools.combinations(sorted(set(k for k in kws if k in vocab)), 2))


def build_keyword_graph(weighting="cosine", top_nb=TOP_NB):
    g = nx.Graph()
    for (a, b), c in pair_c.items():
        if c >= MIN_C:
            w = c if weighting == "count" else c / math.sqrt(kw_freq[a] * kw_freq[b])
            g.add_edge(a, b, weight=w, count=c)
    if top_nb:
        keep = set()
        for n in g:
            for m, _ in sorted(g[n].items(), key=lambda x: -x[1]["weight"])[:top_nb]:
                keep.add(tuple(sorted((n, m))))
        g = g.edge_subgraph(keep).copy()
    for u, v, d in g.edges(data=True):
        d["distance"] = 1 / d["weight"]
    nx.set_node_attributes(g, {n: kw_freq[n] for n in g}, "df")
    return g


def giant_of(g):
    return g.subgraph(max(nx.connected_components(g), key=len)).copy()


cmp = []
for name, (wt, tn) in {"совместная частота": ("count", None),
                       "совместная частота, 10 сильнейших связей": ("count", TOP_NB),
                       "косинус": ("cosine", None),
                       "косинус, 10 сильнейших связей (выбрано)": ("cosine", TOP_NB)}.items():
    g = giant_of(build_keyword_graph(wt, tn))
    comms = nx.community.louvain_communities(g, weight="weight", seed=42)
    cmp.append({"взвешивание": name, "рёбер": g.number_of_edges(), "кластеров": len(comms),
                "модулярность Q (Louvain)": nx.community.modularity(g, comms, weight="weight")})
display(pd.DataFrame(cmp).set_index("взвешивание").round(3))

G = build_keyword_graph()
giant = giant_of(G)
print(f"вершин: {G.number_of_nodes()}, рёбер: {G.number_of_edges()}, "
      f"компонент связности: {nx.number_connected_components(G)}")
print(f"гигантская компонента: {giant.number_of_nodes()} вершин ({giant.number_of_nodes() / G.number_of_nodes():.0%}), "
      f"плотность {nx.density(giant):.3f}, средняя степень {2 * giant.number_of_edges() / giant.number_of_nodes():.1f}, "
      f"коэффициент кластеризации {nx.average_clustering(giant):.3f}")
print(f"средний кратчайший путь (в шагах): {nx.average_shortest_path_length(giant):.2f}, диаметр: {nx.diameter(giant)}")


### 4.1. Кластеры ключевых слов

Сравниваем три алгоритма выделения сообществ. Модулярность показывает, насколько связи сосредоточены внутри кластеров.


In [ ]:
algos = {
    "Louvain": lambda g: nx.community.louvain_communities(g, weight="weight", resolution=1.0, seed=42),
    "жадная модулярность (CNM)": lambda g: nx.community.greedy_modularity_communities(g, weight="weight"),
    "распространение меток": lambda g: list(nx.community.asyn_lpa_communities(g, weight="weight", seed=42)),
}
rows, parts = [], {}
for name, fn in algos.items():
    comms = [set(c) for c in fn(giant)]
    parts[name] = comms
    cov, perf = nx.community.partition_quality(giant, comms)
    sizes = sorted((len(c) for c in comms), reverse=True)
    rows.append({"алгоритм": name, "кластеров": len(comms),
                 "модулярность Q": nx.community.modularity(giant, comms, weight="weight"),
                 "coverage": cov, "performance": perf, "размеры": sizes[:6]})
pd.DataFrame(rows).set_index("алгоритм").round(3)


In [ ]:
communities = sorted(parts["Louvain"], key=len, reverse=True)
node_comm = {n: i for i, c in enumerate(communities) for n in c}
strength = dict(giant.degree(weight="weight"))
summary = []
for i, c in enumerate(communities):
    top = sorted(c, key=lambda n: -kw_freq[n])[:12]
    sub = giant.subgraph(c)
    summary.append({"кластер": i, "размер": len(c),
                    "доля рёбер внутри": sub.size(weight="weight") / sum(strength[n] for n in c) * 2,
                    "главные ключевые слова (по числу статей)": ", ".join(top)})
comm_df = pd.DataFrame(summary).set_index("кластер")
comm_df.round(2)


In [ ]:

top_nodes = sorted(giant, key=lambda n: -kw_freq[n])[:150]
H = giant.subgraph(top_nodes)
pos = nx.spring_layout(H, weight="weight", k=0.45, iterations=300, seed=42)
palette = plt.cm.tab20.colors
fig, ax = plt.subplots(figsize=(15, 11))
nx.draw_networkx_edges(H, pos, ax=ax, alpha=0.12, width=[0.4 + 3 * H[u][v]["weight"] for u, v in H.edges()])
nx.draw_networkx_nodes(H, pos, ax=ax, node_size=[25 * H.nodes[n]["df"] ** 0.9 for n in H],
                       node_color=[palette[node_comm[n] % 20] for n in H], alpha=0.85, linewidths=0)
labels = {n: n for n in sorted(H, key=lambda n: -kw_freq[n])[:70]}
nx.draw_networkx_labels(H, pos, labels, font_size=8, ax=ax)
for i in range(min(len(communities), 12)):
    ax.scatter([], [], color=palette[i % 20], label=f"кластер {i}: {comm_df.iloc[i, 2].split(', ')[0]}")
ax.legend(loc="lower left", fontsize=9, frameon=True)
ax.set_title("Граф ключевых слов Graph RAG (150 самых частых фраз)")
ax.axis("off"); plt.tight_layout(); plt.show()


In [ ]:
CG = nx.Graph()
for i, c in enumerate(communities):
    CG.add_node(i, size=len(c), label="\n".join(sorted(c, key=lambda n: -kw_freq[n])[:3]))
for u, v, d in giant.edges(data=True):
    a, b = node_comm[u], node_comm[v]
    if a != b:
        CG.add_edge(a, b, weight=CG[a][b]["weight"] + d["weight"] if CG.has_edge(a, b) else d["weight"])
pos_c = nx.spring_layout(CG, weight="weight", k=1.6, seed=7)
wmax = max(w for *_, w in CG.edges(data="weight"))
fig, ax = plt.subplots(figsize=(13, 9))
nx.draw_networkx_edges(CG, pos_c, ax=ax, width=[0.3 + 7 * w / wmax for *_, w in CG.edges(data="weight")],
                       alpha=0.35, edge_color="#555555")
nx.draw_networkx_nodes(CG, pos_c, ax=ax, node_size=[35 * CG.nodes[n]["size"] for n in CG],
                       node_color=[palette[n % 20] for n in CG], alpha=0.9, linewidths=0)
nx.draw_networkx_labels(CG, pos_c, {n: f"{n}\n{CG.nodes[n]['label']}" for n in CG}, font_size=8.5, ax=ax)
ax.set_title("Граф кластеров ключевых слов: размер — число фраз в кластере, толщина ребра — сила связи")
ax.axis("off"); plt.tight_layout(); plt.show()


### 4.2. Интерпретация кластеров

По ведущим фразам в `comm_df` определяем темы сообществ. Высокая модулярность показывает разделение графа, но сами темы нужно проверить по содержанию.


## 5. Центральность ключевых слов

Считаем шесть мер на крупнейшей связной компоненте:

- **degree** — число соседей;
- **strength** — сумма весов связей;
- **betweenness** — роль моста между темами;
- **eigenvector** — связи с важными вершинами;
- **closeness** — близость к остальным вершинам;
- **PageRank** — важность с учётом весов связей.

Для кратчайших путей используем расстояние `1 / вес`.


In [ ]:
cent = pd.DataFrame({
    "degree": nx.degree_centrality(giant),
    "strength": pd.Series(strength) / max(strength.values()),
    "betweenness": nx.betweenness_centrality(giant, weight="distance", seed=42),
    "eigenvector": nx.eigenvector_centrality(giant, weight="weight", max_iter=2000),
    "closeness": nx.closeness_centrality(giant, distance="distance"),
    "pagerank": nx.pagerank(giant, weight="weight"),
})
cent["статей"] = [kw_freq[n] for n in cent.index]
cent["кластер"] = [node_comm[n] for n in cent.index]
top10 = pd.DataFrame({m: cent[m].sort_values(ascending=False).head(10).index.tolist() for m in
                      ["degree", "betweenness", "eigenvector", "closeness", "pagerank"]})
top10.index = range(1, 11)
top10


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
corr = cent[["degree", "strength", "betweenness", "eigenvector", "closeness", "pagerank"]].corr("spearman")
im = ax[0].imshow(corr, cmap="Blues", vmin=0, vmax=1)
ax[0].set_xticks(range(6), corr.columns, rotation=45, ha="right"); ax[0].set_yticks(range(6), corr.columns)
for i in range(6):
    for j in range(6):
        ax[0].text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax[0].set_title("Корреляция Спирмена между центральностями"); ax[0].grid(False)

ax[1].scatter(cent.degree, cent.betweenness, s=12, alpha=0.5, color="#4a6fa5")
bridges = (cent.betweenness / cent.betweenness.max() - cent.degree / cent.degree.max()).nlargest(8).index
for n in list(cent.degree.nlargest(5).index) + list(bridges):
    ax[1].annotate(n, (cent.degree[n], cent.betweenness[n]), fontsize=8, xytext=(3, 2), textcoords="offset points")
ax[1].set_xlabel("degree centrality"); ax[1].set_ylabel("betweenness centrality")
ax[1].set_title("Популярность vs посредничество")
plt.tight_layout(); plt.show()
print("фразы-«мосты» (посредничество заметно выше, чем степень):", ", ".join(bridges))


### 5.1. Интерпретация центральностей

Сравниваем рейтинги и корреляции мер: они описывают разные роли терминов в графе. Результаты сохраняем в `centralities.csv`.


## 6. Граф публикаций и поиск близких публикаций

Вершины — статьи. Связываем их по общим ключевым фразам, вес ребра — число совпадений.

Похожие статьи выбираем по весу связи, при равенстве — по коэффициенту Жаккара. Для текстового запроса находим совпавшие фразы, задаём начальные веса по IDF и запускаем персонализированный PageRank.


In [ ]:
kw_sets = [set(k) for k in df.keywords]
inv = defaultdict(set)
for i, ks in enumerate(kw_sets):
    for k in ks:
        inv[k].add(i)

P = nx.Graph()
P.add_nodes_from((i, {"title": t}) for i, t in enumerate(df.title))
shared = Counter()
for k, docs in inv.items():
    shared.update(itertools.combinations(sorted(docs), 2))
for (a, b), w in shared.items():
    P.add_edge(a, b, weight=w, jaccard=w / len(kw_sets[a] | kw_sets[b]))

deg = np.array([d for _, d in P.degree()])
print(f"публикаций: {P.number_of_nodes()}, рёбер: {P.number_of_edges()}, изолированных: {(deg == 0).sum()}")
print(f"средняя степень: {deg.mean():.1f}; рёбер с весом ≥2: {sum(1 for *_, w in P.edges(data='weight') if w >= 2)}")
print(f"распределение весов рёбер: {Counter(w for *_, w in P.edges(data='weight'))}")


In [ ]:
def similar_papers(i, k=5):
    nb = sorted(P[i].items(), key=lambda x: (-x[1]["weight"], -x[1]["jaccard"]))[:k]
    return pd.DataFrame([{"вес": d["weight"], "Жаккар": round(d["jaccard"], 2), "название": df.title[j],
                          "общие ключевые слова": ", ".join(sorted(kw_sets[i] & kw_sets[j]))}
                         for j, d in nb])


i0 = int(df.index[df.title.str.contains("From Local to Global", case=False)].min()
         if df.title.str.contains("From Local to Global", case=False).any() else 0)
print("исходная статья:", df.title[i0]); print("ключевые слова:", df.keywords[i0])
similar_papers(i0)


### 6.1. Косинусное сходство публикаций

Представляем название и аннотацию в виде TF-IDF-вектора и сравниваем статьи по косинусу:

$$\operatorname{cos}(a,b)=\frac{a\cdot b}{\lVert a\rVert\lVert b\rVert}.$$

Сравниваем четыре варианта: общие ключевые фразы, косинус текста, косинус ключевых фраз и их гибрид. TF-IDF снижает вес частых слов; популярность статьи в ранжировании не используем. Метод учитывает общие слова, поэтому может пропускать перефразированные тексты.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Сравниваем названия и аннотации.
content = (df.title.fillna("") + ". " + df.abstract.fillna("")).tolist()
vectorizer_text = TfidfVectorizer(
    stop_words="english", ngram_range=(1, 2), min_df=2,
    max_df=0.85, sublinear_tf=True, norm="l2", dtype=np.float32
)
X_text = vectorizer_text.fit_transform(content)

# Ключевую фразу считаем одним признаком.
vectorizer_kw = TfidfVectorizer(
    tokenizer=str.split, preprocessor=None, token_pattern=None,
    lowercase=False, binary=True, use_idf=True, norm="l2", dtype=np.float32
)
keyword_documents = [" ".join("__".join(p.split()) for p in sorted(set(keys))) for keys in df.keywords]
X_kw = vectorizer_kw.fit_transform(keyword_documents)


def cosine_row(matrix, idx):
    return (matrix @ matrix[idx].T).toarray().ravel()


def comparable_neighbors(idx, top_k=5, text_weight=0.75):
    if not 0 <= idx < len(df):
        raise IndexError("Статья вне корпуса")
    if not 0 <= text_weight <= 1:
        raise ValueError("text_weight должен быть в [0, 1]")
    txt = cosine_row(X_text, idx)
    kw = cosine_row(X_kw, idx)
    shared = np.fromiter((len(kw_sets[idx] & ks) for ks in kw_sets), dtype=float, count=len(df))
    hybrid = text_weight * txt + (1 - text_weight) * kw
    methods = {"общие ключевые слова": shared, "косинус: текст": txt,
               "косинус: ключевые фразы": kw, "гибрид косинусов": hybrid}
    rows = []
    for method, scores in methods.items():
        order = np.argsort(-scores, kind="stable")
        # Саму статью из выдачи убираем.
        order = [int(j) for j in order if j != idx and scores[j] > 0][:top_k]
        for pos, j in enumerate(order, 1):
            rows.append({"метод": method, "место": pos, "индекс": j,
                         "cos_text": round(float(txt[j]), 4),
                         "cos_keywords": round(float(kw[j]), 4),
                         "общих ключевых": int(shared[j]),
                         "та же категория": bool(df.primary_category.iloc[idx] == df.primary_category.iloc[j]),
                         "название": df.title.iloc[j]})
    return pd.DataFrame(rows)

print("TF-IDF размерность (статьи × текстовые признаки):", X_text.shape)
print("TF-IDF ключевых фраз:", X_kw.shape)
print("Сопоставляем статью:", df.title.iloc[i0])
display(comparable_neighbors(i0, top_k=5))


### 6.2. Примеры похожих статей

Проверяем несколько заранее выбранных статей. Чем ближе косинус к 1, тем ближе их TF-IDF-векторы. Совпадение категории arXiv помогает оценить результат, но не гарантирует релевантность.


In [ ]:
examples = sorted(set([i0, N_DOCS // 4, N_DOCS // 2, 3 * N_DOCS // 4]))
for idx in examples:
    print("\nИСХОДНАЯ СТАТЬЯ:", df.title.iloc[idx])
    display(comparable_neighbors(idx, top_k=3))


In [ ]:
IDF = {k: math.log(N_DOCS / len(d)) for k, d in inv.items()}


def query_keywords(q):
    doc = nlp(normalize_text(q))
    found = {p for p in candidates(doc) if p in inv}
    words = [RESTORE.get(w.lower_, w.lemma_.lower()) for w in doc if w.is_alpha and not w.is_stop]
    for n in (1, 2):
        for j in range(len(words) - n + 1):
            p = " ".join(words[j:j + n])
            if p in inv:
                found.add(p)
    return found


def search(q, k=10, alpha=0.3, use_graph=True):
    kws = query_keywords(q)
    seeds = Counter()
    for kw in kws:
        for d in inv[kw]:
            seeds[d] += IDF[kw]
    if not seeds:
        return pd.DataFrame(), kws
    if use_graph:
        scores = nx.pagerank(P, alpha=alpha, personalization=dict(seeds), weight="weight")
    else:
        scores = seeds
    best = sorted(scores, key=lambda d: -scores[d])[:k]
    res = pd.DataFrame([{"скор": scores[d], "прямое совпадение": d in seeds, "название": df.title[d],
                         "совпавшие ключевые слова": ", ".join(sorted(kw_sets[d] & kws)), "год": df.published[d].year}
                        for d in best])
    return res, kws


for q in ["reducing hallucinations in medical question answering with knowledge graphs",
          "multi-hop reasoning over graph structure",
          "community summaries for global sensemaking questions over a large corpus"]:
    res, kws = search(q)
    print(f"\nзапрос: «{q}»\nраспознанные ключевые слова: {sorted(kws)}")
    display(res.round(4))


## 7. Тестирование поиска

### 7.1. Поиск по названию

Берём названия 200 случайных статей как запросы, а индекс строим только по аннотациям. Считаем Hit@1, Hit@10 и MRR@10 для совпадений ключевых фраз, PageRank с разными α и TF-IDF.

Также сравниваем долю соседей одной категории arXiv с долей у случайных пар.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel


def keywords_abstract_only(i):
    tf = Counter(cand_abs[i])
    scores = {p: (1 + math.log(c)) * math.log(N_DOCS / doc_freq[p])
              for p, c in tf.items() if doc_freq[p] >= MIN_KW_DF and p not in too_common}
    return {p for p, _ in sorted(scores.items(), key=lambda x: -x[1])[:TOP_K]}

kw_abs = [keywords_abstract_only(i) for i in range(N_DOCS)]
inv_a = defaultdict(set)
for i, ks in enumerate(kw_abs):
    for k in ks:
        inv_a[k].add(i)
IDF_a = {k: math.log(N_DOCS / len(d)) for k, d in inv_a.items()}
P_a = nx.Graph(); P_a.add_nodes_from(range(N_DOCS))
sh = Counter()
for k, docs in inv_a.items():
    sh.update(itertools.combinations(sorted(docs), 2))
P_a.add_weighted_edges_from((a, b, w) for (a, b), w in sh.items())

tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xt = tfidf.fit_transform(df.abstract)

test_ids = random.Random(0).sample(range(N_DOCS), 200)


def rr_at10(rank, i):
    rank = list(rank)[:10]
    return 1 / (rank.index(i) + 1) if i in rank else 0


variants = {"только совпадение ключевых слов": None, "граф, PPR α=0.85": 0.85,
            "граф, PPR α=0.5": 0.5, "граф, PPR α=0.3": 0.3}
res_rows = {name: [] for name in list(variants) + ["TF-IDF по аннотациям"]}
for i in test_ids:
    q = df.title[i]
    kws = {k for k in query_keywords(q) if k in inv_a}
    seeds = Counter()
    for kw in kws:
        for d in inv_a[kw]:
            seeds[d] += IDF_a[kw]
    for name, alpha in variants.items():
        if not seeds:
            res_rows[name].append(0); continue
        sc = seeds if alpha is None else nx.pagerank(P_a, alpha=alpha, personalization=dict(seeds), weight="weight")
        res_rows[name].append(rr_at10(sorted(sc, key=lambda d: -sc[d]), i))
    sims = linear_kernel(tfidf.transform([q]), Xt).ravel()
    res_rows["TF-IDF по аннотациям"].append(rr_at10(np.argsort(-sims), i))

eval_df = pd.DataFrame({name: {"Hit@1": np.mean([r == 1 for r in rr]), "Hit@10": np.mean([r > 0 for r in rr]),
                               "MRR@10": np.mean(rr)} for name, rr in res_rows.items()}).T
print(f"запросов без единого распознанного ключевого слова: "
      f"{sum(1 for i in test_ids if not ({k for k in query_keywords(df.title[i]) if k in inv_a})) } из 200")
eval_df.round(3)


In [ ]:
cat = df.primary_category.values
same_nb, same_rand = [], []
rng = np.random.default_rng(0)
for i in range(N_DOCS):
    nb = sorted(P[i].items(), key=lambda x: -x[1]["weight"])[:5]
    if nb:
        same_nb += [cat[i] == cat[j] for j, _ in nb]
        same_rand += [cat[i] == cat[j] for j in rng.integers(0, N_DOCS, len(nb))]
print(f"топ-5 соседей в графе той же основной категории arXiv: {np.mean(same_nb):.1%}; "
      f"случайная пара: {np.mean(same_rand):.1%}")

years = df.published.dt.year.values
gap_nb = [abs(years[i] - years[j]) for i in range(N_DOCS) for j, _ in sorted(P[i].items(), key=lambda x: -x[1]["weight"])[:5]]
print(f"средняя разница в годах публикации у соседей: {np.mean(gap_nb):.2f} против "
      f"{np.mean(np.abs(years[rng.integers(0, N_DOCS, 5000)] - years[rng.integers(0, N_DOCS, 5000)])):.2f} у случайных пар")


Поиск по названию проверяет, удаётся ли найти известную статью. Для оценки реальных запросов нужны отдельные примеры с разметкой релевантности.


### 7.2. Сравнение похожих статей

Для одних и тех же 200 статей находим до пяти соседей каждым методом. Саму исходную статью исключаем.

Сравниваем число соседей, совпадение категории, Jaccard ключевых фраз, косинус текста и число разных найденных статей. Косинус текста выгоден текстовому методу, поэтому смотрим на все показатели вместе.


In [ ]:
eval_ids = np.random.default_rng(42).choice(N_DOCS, size=min(200, N_DOCS), replace=False)
method_names = ["общие ключевые слова", "косинус: текст", "косинус: ключевые фразы", "гибрид косинусов"]
records = []
for i in eval_ids:
    rankings = comparable_neighbors(int(i), top_k=5)
    if rankings.empty:
        continue
    text_scores = cosine_row(X_text, int(i))
    for method in method_names:
        neighbors = rankings.loc[rankings["метод"] == method, "индекс"].tolist()
        if not neighbors:
            continue
        jac = [len(kw_sets[i] & kw_sets[j]) / len(kw_sets[i] | kw_sets[j])
               if kw_sets[i] | kw_sets[j] else 0.0 for j in neighbors]
        records.append({"метод": method, "источник": int(i),
                        "соседей": len(neighbors),
                        "одна категория": np.mean(df.primary_category.iloc[neighbors].values == df.primary_category.iloc[i]),
                        "Jaccard фраз": np.mean(jac),
                        "косинус текста": np.mean(text_scores[neighbors]),
                        "список индексов": neighbors})

by_method = pd.DataFrame(records)
summary = by_method.groupby("метод").agg(
    **{"среднее соседей": ("соседей", "mean"),
       "одна категория": ("одна категория", "mean"),
       "Jaccard фраз": ("Jaccard фраз", "mean"),
       "косинус текста": ("косинус текста", "mean")}
)
summary["разные соседи"] = by_method.groupby("метод")["список индексов"].apply(
    lambda s: len(set(j for ids in s for j in ids)))
summary = summary.reindex(method_names)
display(summary.round(3))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
summary["одна категория"].plot(kind="bar", ax=ax[0], title="Совпадение основной категории arXiv", ylabel="доля")
summary["косинус текста"].plot(kind="bar", ax=ax[1], title="Среднее TF-IDF косинусное сходство", ylabel="сходство")
for a in ax:
    a.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()


### 7.3. Интерпретация сравнения

Граф учитывает общие ключевые фразы, текстовый косинус — слова названия и аннотации. Косинус фраз сравнивает термины целиком. Гибрид объединяет два косинуса с весами 0.75 и 0.25; эти веса заданы заранее.

Метрики показывают близость и разнообразие результатов. Чтобы оценить релевантность, нужна ручная проверка; лучший метод выбираем по результатам запуска.


## 8. Итоги

Сохраняем корпус, графы, центральности и сообщества в `graph_artifacts`. Размеры графов берём из текущего запуска.


In [ ]:
from pathlib import Path
import json
OUT=Path('graph_artifacts');OUT.mkdir(exist_ok=True)
cent.to_csv(OUT/'centralities.csv')
comm_df.to_csv(OUT/'communities.csv')
df.to_json(OUT/'articles_with_keywords.jsonl',orient='records',lines=True,force_ascii=False,date_format='iso')
nx.to_pandas_edgelist(G).to_csv(OUT/'keyword_edges.csv',index=False)
nx.to_pandas_edgelist(P).to_csv(OUT/'publication_edges.csv',index=False)
graph_summary={'articles':len(df),'keyword_nodes':G.number_of_nodes(),'keyword_edges':G.number_of_edges(),
 'giant_nodes':giant.number_of_nodes(),'publication_nodes':P.number_of_nodes(),'publication_edges':P.number_of_edges(),
 'date_cutoff':CUTOFF,'communities':len(communities)}
(OUT/'summary.json').write_text(json.dumps(graph_summary,ensure_ascii=False,indent=2))
print('FRESH_GRAPH_EXPORT',graph_summary)
display(cent.sort_values('betweenness',ascending=False).head(5))


## Выводы

В сохранённом запуске получено 1082 статьи до 06.11.2026. Граф терминов содержит 733 вершины, 3130 рёбер и 14 сообществ; граф публикаций — 142866 рёбер без изолированных статей.

На 200 запросах TF-IDF дал Hit@10 = 0.910 и MRR@10 = 0.811. Совпадения ключевых фраз — 0.325 и 0.181; PPR с α=0.85 — 0.215 и 0.111. Для 7 запросов ключевые фразы не нашлись. В этой проверке текстовый поиск оказался точнее графового.

У соседних статей категория совпала в 26.0% случаев, у случайных пар — в 20.1%. Это признак тематической близости. Сравнение новых косинусных методов появится после запуска добавленных ячеек.


## Как читать результат

В сохранённом запуске: 1081 публикация, 732 термина и 15 сообществ. Это анализ графа документов и терминов, а не реализация Microsoft GraphRAG. Ограничение даты корпуса фиксировано в конфигурации.

Состав корпуса зависит от даты и ответа arXiv. Кластеры и центральности описывают собранный корпус, а не всю научную область.

Сохранённая история исходного эксперимента находится отдельно в `results/recorded_metrics.json`. Полное GPU-обучение при подготовке этой версии не выполнялось.